In [ ]:
from collections import defaultdict

import pandas
import json

class ViliaEconomy:

    def __init__(self, fmg_json):
        pack         = fmg_json["pack"]
        self.routes  = pack["routes"]
        self.biomes  = fmg_json["biomesData"]
        self.states  = [s for s in pack["states"]    if isinstance(s, dict)]
        self.provinces = [p for p in pack["provinces"] if isinstance(p, dict)]
        self.cells   = [c for c in pack["cells"]     if isinstance(c, dict)]

        self.year = fmg_json.get("settings", {}).get("options", {}).get("year", 1)

        

def load_from_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return ViliaEconomy(data)

data = load_from_file("ViliaFull.json")

BASE_DEMAND = {
    "food": 0.5,
    "stone": 0.2,
    "gold": 0.1,
}
RESOURCES = ["food","stone","gold"]

def calculate_demand(self):
    out = {}
    for s in self.states:
        # FIX: compute each state's demand independently (no cross-state accumulation)
        state_demand = {}
        for resource in RESOURCES:
            pop = s.get("rural", 0) + s.get("urban", 0)
            state_demand[resource] = pop * BASE_DEMAND[resource]
        out[s["name"]] = state_demand
    return out

print(calculate_demand(data))



{'Neutrals': {'food': 223.29589930176735, 'stone': 89.31835972070695, 'gold': 44.659179860353476}, 'Monch': {'food': 302.2046013498306, 'stone': 120.88184053993226, 'gold': 60.44092026996613}, 'Bacseslamia': {'food': 2946.7630981134175, 'stone': 1178.705239245367, 'gold': 589.3526196226835}, 'Kouria': {'food': 8101.5336991728545, 'stone': 3240.613479669142, 'gold': 1620.306739834571}, 'Memein': {'food': 2130.1737527983187, 'stone': 852.0695011193275, 'gold': 426.03475055966373}, 'Ingne': {'food': 2752.0686494166853, 'stone': 1100.827459766674, 'gold': 550.413729883337}, 'Ardonia': {'food': 1449.9021475809814, 'stone': 579.9608590323926, 'gold': 289.9804295161963}, 'Loshin': {'food': 8667.679304345727, 'stone': 3467.071721738291, 'gold': 1733.5358608691456}, 'Degyesia': {'food': 4147.284805742741, 'stone': 1658.9139222970964, 'gold': 829.4569611485482}, 'Anaia': {'food': 6252.690898558139, 'stone': 2501.076359423256, 'gold': 1250.538179711628}, 'Ogskifta': {'food': 2176.066747987747, 's

In [59]:
GRAIN_BIOME = {
    0: 0.00, 1: 0.05, 2: 0.05,  3: 0.80,  4: 1.00,
    5: 1.15, 6: 1.15, 7: 1.44,  8: 1.34,  9: 0.40,
   10: 0.20, 11: 0.06, 12: 1.12,
}

STONE_BIOME = {
    0: 0.00, 1: 0.13, 2: 1.15,  3: 0.50,  4: 0.40,
    5: 0.12, 6: 0.23, 7: 0.08,  8: 0.21,  9: 0.40,
   10: 0.70, 11: 0.06, 12: 0.30,
}
GOLD_BIOME = {
    0: 0.00, 1: 0.06, 2: 0.65,  3: 0.25,  4: 0.20,
    5: 0.06, 6: 0.11, 7: 0.04,  8: 0.1,  9: 0.20,
   10: 0.35, 11: 0.03, 12: 0.15,
}
#unused for now, but could be used to calculate the total production of a cell based on its biome and other factors
#also because it crashes the production causing mass death, so we need to tune it before we can use it
def calculate_cell_capacity(c, food_per_person=1):
    # for c in self.cells:
    if c.get("h", 0) > 20:  # Only cells with elevation above 20 can produce
        biome_id = c["biome"]
        grain_yield = GRAIN_BIOME.get(biome_id, 0)
        # Bonus for being near a river or coast
        if c.get("river", False):
            grain_yield *= 1.2
        if c.get("coast", False):
            grain_yield *= 1.1
        if c.get("pop") > 0:
            grain_yield *= 1 + (c["pop"] / 1)  # Bonus for population

        grain_yield = grain_yield - c.get("height", 0) * 0.01
        return grain_yield/food_per_person

print(data)
def labor_efficiency(pop, capacity):
    base = pop / (pop + capacity)

    # overcrowding penalty
    pressure = pop / capacity
    penalty = 1 / (1 + max(0, pressure - 1))

    return base * penalty



#create the max of stone for each cell based on its height and biome, with a bonus for being near a river or coast
for c in data.cells:
    if c.get("h", 0) > 20:  # Only cells with elevation above 20 can produce
        biome_id = c["biome"]
        stone_yield = STONE_BIOME.get(biome_id, 0)
        gold_yield = GOLD_BIOME.get(biome_id, 0)
        # Bonus for being near a river or coast
        if c.get("river", False):
            stone_yield *= 1.2
            gold_yield *= 1.6
        if c.get("coast", False):
            stone_yield *= 0.8
            gold_yield *= 0.8
        if c.get("pop") > 0:
            stone_yield *= 1 + (c["pop"] / 1)  # Bonus for population
            gold_yield *= 1 + (c["pop"] / 1)  # Bonus for population

        

        stone_yield *= 1 + c.get("height", 0)/1000
        gold_yield *= 1 + c.get("height", 0)/1000
        c["stone_max"] = stone_yield
        c["gold_max"] = gold_yield
    else:
        c["stone_max"] = 0
        c["gold_max"] = 0
    c["stone_reserve"] = c["stone_max"]*10
    c["gold_reserve"] = c["gold_max"]*10

#print the aggerated stone max for each state, and the total stone max
state_stone_max = defaultdict(float)

print("Total stone max:", sum(c["stone_max"] for c in data.cells))
print("stone Reserve:", sum(c["stone_reserve"] for c in data.cells))
print("Total gold max:", sum(c["gold_max"] for c in data.cells))
print("gold Reserve:", sum(c["gold_reserve"] for c in data.cells))



Total stone max: 45623.3965183501
stone Reserve: 456233.96518350096
Total gold max: 22318.90950459942
gold Reserve: 223189.0950459942


In [60]:
#cells can only produce when elevation is above 20
#define a carry capacity for each cell based on its elevation and biome, with a bonus if it is near a river or coast



def calculate_production(self):
    # food production is based on the biome, with a bonus for being near a river or coast, and a penalty for being at high elevation
    
    def calculate_food_production():
        cell_production = {}  # state_id -> total food
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 20 can produce
                biome_id = c["biome"]
                grain_yield = GRAIN_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    grain_yield *= 1.2
                if c.get("coast", False):
                    grain_yield *= 1.1
                if c.get("pop", 0) > 0:
                    grain_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                grain_yield = grain_yield - c.get("height", 0) * 0.01  # Penalty for height
                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + grain_yield

        # Roll up cell totals to state names
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"food": cell_production.get(state_id, 0)}
        return out
    
    def calculate_stone_production():
        cell_production = {}  # state_id -> total stone
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                stone_yield = STONE_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    stone_yield *= 1.2
                if c.get("coast", False):
                    stone_yield *= 0.8
                if c.get("pop", 0) > 0:
                    stone_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                stone_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("stone_reserve", 0)
                actual = min(stone_yield, reserve)

                # reduce reserve
                c["stone_reserve"] = reserve - actual

                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + actual

        # Roll up cell totals to state names (FIX: no stale state_id keys to delete)
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"stone": cell_production.get(state_id, 0)}
        return out
    
    def calculate_gold_production():
        cell_production = {}  # state_id -> total gold
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                gold_yield = GOLD_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    gold_yield *= 1.2
                if c.get("coast", False):
                    gold_yield *= 0.8
                if c.get("pop", 0) > 0:
                    gold_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                gold_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("gold_reserve", 0)
                actual = min(gold_yield, reserve)

                # reduce reserve
                c["gold_reserve"] = reserve - actual

                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + actual

        # Roll up cell totals to state names (FIX: no stale state_id keys to delete)
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"gold": cell_production.get(state_id, 0)}
        return out

        
    grain=calculate_food_production()
    stone=calculate_stone_production()
    gold=calculate_gold_production()
    combined = {}
    for state in self.states:
        name = state["name"]
        combined[name] = {
            "food": grain.get(name, {}).get("food", 0),
            "stone": stone.get(name, {}).get("stone", 0),
            "gold": gold.get(name, {}).get("gold", 0)
        }

    return combined

print(calculate_production(data))

print(calculate_demand(data))



{'Neutrals': {'food': 373.2485845232009, 'stone': 64.43628807663917, 'gold': 31.001820036768912}, 'Monch': {'food': 654.4711554813383, 'stone': 156.44904204368584, 'gold': 76.33596051216124}, 'Bacseslamia': {'food': 7249.494476523395, 'stone': 663.707453635931, 'gold': 331.477784821391}, 'Kouria': {'food': 19337.44332794192, 'stone': 3991.057343716628, 'gold': 1939.7862523472336}, 'Memein': {'food': 4981.6098310554025, 'stone': 1337.6276315712942, 'gold': 656.344068765044}, 'Ingne': {'food': 6707.611832284923, 'stone': 1132.1011217415332, 'gold': 546.17720287621}, 'Ardonia': {'food': 3870.7875650924434, 'stone': 754.2777004754538, 'gold': 373.1724197575449}, 'Loshin': {'food': 20256.496756878685, 'stone': 4608.582587457596, 'gold': 2260.8315666830517}, 'Degyesia': {'food': 10530.922603585715, 'stone': 1352.37839101553, 'gold': 656.8351879727841}, 'Anaia': {'food': 15093.39989011526, 'stone': 2981.247750004529, 'gold': 1434.2660560053575}, 'Ogskifta': {'food': 5523.474478221552, 'stone'

In [65]:
#calc the demand-production gap for each state, and print it out in a nice format
def calculate_gap(self):
    demand = calculate_demand(self)
    production = calculate_production(self)

    out = {}
    for state in self.states:
        name = state["name"]
        out[name] = {
            "food_gap": production.get(name, {}).get("food", 0)-demand.get(name, {}).get("food", 0) ,
            "stone_gap": production.get(name, {}).get("stone", 0)-demand.get(name, {}).get("stone", 0),
            "gold_gap": production.get(name, {}).get("gold", 0)-demand.get(name, {}).get("gold", 0)
        }
    return out
print(calculate_gap(data))

#also calc global gap
def calculate_global_gap(self):
    demand = calculate_demand(self)
    production = calculate_production(self)

    out = {
        "food_gap": sum(demand.get(s["name"], {}).get("food", 0) for s in self.states) - sum(production.get(s["name"], {}).get("food", 0) for s in self.states),
        "stone_gap": sum(demand.get(s["name"], {}).get("stone", 0) for s in self.states) - sum(production.get(s["name"], {}).get("stone", 0) for s in self.states),
        "gold_gap": sum(demand.get(s["name"], {}).get("gold", 0) for s in self.states) - sum(production.get(s["name"], {}).get("gold", 0) for s in self.states)
    }
    return out
print(calculate_global_gap(data))
print("Total stone max:", sum(c["stone_max"] for c in data.cells))
print("Stone Reserve:", sum(c["stone_reserve"] for c in data.cells))
print("Total gold max:", sum(c["gold_max"] for c in data.cells))
print("Gold Reserve:", sum(c["gold_reserve"] for c in data.cells))


{'Neutrals': {'food_gap': 149.95268522143357, 'stone_gap': -24.88207164406778, 'gold_gap': -13.657359823584564}, 'Monch': {'food_gap': 352.2665541315077, 'stone_gap': 35.56720150375358, 'gold_gap': 15.895040242195108}, 'Bacseslamia': {'food_gap': 4302.731378409977, 'stone_gap': -514.997785609436, 'gold_gap': -257.8748348012925}, 'Kouria': {'food_gap': 11235.909628769066, 'stone_gap': 750.4438640474859, 'gold_gap': 319.4795125126625}, 'Memein': {'food_gap': 2851.436078257084, 'stone_gap': 485.5581304519667, 'gold_gap': 230.30931820538024}, 'Ingne': {'food_gap': 3955.543182868238, 'stone_gap': 31.273661974859124, 'gold_gap': -4.2365270071270515}, 'Ardonia': {'food_gap': 2420.885417511462, 'stone_gap': 174.3168414430612, 'gold_gap': 83.1919902413486}, 'Loshin': {'food_gap': 11588.817452532958, 'stone_gap': 1141.5108657193045, 'gold_gap': 527.2957058139061}, 'Degyesia': {'food_gap': 6383.637797842975, 'stone_gap': -306.53553128156636, 'gold_gap': -172.62177317576413}, 'Anaia': {'food_gap':

In [66]:
#figure out the tradebetweenburgsandtheroutestheyuse

In [62]:
# #make a plot showing the production and demand for each state, and the gap between them
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 6))
# plt.bar([s["name"] for s in data.states], [calculate_production(data).get(s["name"], {}).get("food", 0) for s in data.states], label="Food Production")
# plt.bar([s["name"] for s in data.states], [calculate_demand(data).get(s["name"], {}).get("food", 0) for s in data.states], label="Food Demand", alpha=0.7)
# plt.bar([s["name"] for s in data.states], [calculate_gap(data).get(s["name"], {}).get("food_gap", 0) for s in data.states], label="Food Gap", alpha=0.5)
# plt.xticks(rotation=45)
# plt.legend()
# plt.title("Food Production, Demand, and Gap by State")
# plt.xlabel("State")
# plt.ylabel("Amount")
# plt.tight_layout()
# plt.show()

# #plot stone total and gap
# plt.figure(figsize=(12, 6))
# plt.bar([s["name"] for s in data.states], [calculate_production(data).get(s["name"], {}).get("stone", 0) for s in data.states], label="Stone Production")
# plt.bar([s["name"] for s in data.states], [calculate_demand(data).get(s["name"], {}).get("stone", 0) for s in data.states], label="Stone Demand", alpha=0.7)
# plt.bar([s["name"] for s in data.states], [calculate_gap(data).get(s["name"], {}).get("stone_gap", 0) for s in data.states], label="Stone Gap", alpha=0.5)
# plt.xticks(rotation=45)
# plt.legend()
# plt.title("Stone Production, Demand, and Gap by State")
# plt.xlabel("State")
# plt.ylabel("Amount")
# plt.tight_layout()
# plt.show()